# Notebook 01: Exploração de Dados Demográficos — IBGE

**Projeto:** `migracao-venezuelana-oeste-sc`  
**Autores:** Leonardo Rafael Santos Leitão e Vicente Neves da Silva Ribeiro (UFFS)  
**Data:** Abril de 2026

---

## Objetivo

Este notebook explora dados demográficos reais do IBGE para o Oeste de Santa Catarina, com foco na população venezuelana. As análises incluem:

1. Carregamento e inspeção dos dados reais baixados pelo pipeline;
2. Pirâmide etária da população venezuelana (stub simulado com estrutura realista);
3. Série temporal da população estimada dos municípios do Oeste;
4. Visualização espacial/comparativa da distribuição populacional.

> **Nota metodológica:** Os dados de população venezuelana são um *stub/placeholder* funcional, pois o IBGE não disponibiliza nacionalidade via SIDRA (tabela 9606) e os microdados do Censo 2022 ainda não foram liberados. A estrutura etária segue perfil demográfico documentado na literatura sobre migração venezuelana no Brasil.

## 1. Configuração do Ambiente

In [ ]:
# -----------------------------------------------------------------------------
# Bibliotecas padrão
# -----------------------------------------------------------------------------
import os
import sys
from pathlib import Path

# Diretórios do projeto
PROJECT_ROOT = Path(".").resolve().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# -----------------------------------------------------------------------------
# Bibliotecas científicas e de dados
# -----------------------------------------------------------------------------
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# Bibliotecas de visualização
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

try:
    import geobr
    import geopandas as gpd
    HAS_GEO = True
except ImportError:
    HAS_GEO = False
    print("[AVISO] geobr/geopandas não instalado. Visualização espacial desabilitada.")

# Configurações globais
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

print(f"Diretório do projeto: {PROJECT_ROOT}")
print(f"Diretório de figuras: {FIGURES_DIR}")
print(f"Geo libs disponíveis: {HAS_GEO}")

## 2. Carregamento dos Dados Reais

In [ ]:
# -----------------------------------------------------------------------------
# 2.1 População estimada (série temporal 2018-2024)
# -----------------------------------------------------------------------------
pop_path = DATA_RAW / "ibge" / "populacao_estimada_2018_2024.parquet"
df_pop = pd.read_parquet(pop_path)
print("População Estimada – dimensões:", df_pop.shape)
display(df_pop.head())

# -----------------------------------------------------------------------------
# 2.2 Stub de migrantes venezuelanos (Censo 2022)
# -----------------------------------------------------------------------------
mig_path = DATA_RAW / "ibge" / "censo2022_migrantes_stub.parquet"
df_mig = pd.read_parquet(mig_path)
print("\nMigrantes Venezuelanos (stub) – dimensões:", df_mig.shape)
display(df_mig.head())
print(f"Total estimado de venezuelanos no Oeste de SC: {df_mig['populacao'].sum()}")

# -----------------------------------------------------------------------------
# 2.3 Totais reais por sexo (Censo 2022) – referência
# -----------------------------------------------------------------------------
total_path = DATA_RAW / "ibge" / "censo2022_total_sexo.parquet"
df_total = pd.read_parquet(total_path)
print("\nPopulação Total por Sexo – dimensões:", df_total.shape)
display(df_total.head())

## 3. Pirâmide Etária Comparativa

Construímos pirâmides etárias para a população venezuelana (stub) e a população total do Oeste de SC.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

# Dados reais processados da RI de Chapecó
df = pd.read_parquet('../data/processed/ibge_populacao_estimada_oeste_sc.parquet')
df = df.dropna(subset=['populacao'])
print(f'Municípios: {df["codigo_ibge_7d"].nunique()}, Registros: {len(df)}')
df.head()

## 4. Série Temporal da População Estimada

Evolução da população estimada dos municípios do Oeste de SC (2018-2024).

In [ ]:
# Série temporal agregada da RI de Chapecó
resumo = df.groupby('ano')['populacao'].sum().reset_index()
plt.figure(figsize=(10,5))
plt.plot(resumo['ano'], resumo['populacao']/1e6, marker='o', color='#2E86AB', linewidth=2)
plt.title('População Estimada — Região Intermediária de Chapecó', fontsize=14)
plt.ylabel('Milhões de habitantes')
plt.xlabel('Ano')
plt.xticks(resumo['ano'])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/01_serie_temporal_ri_chapeco.png', dpi=300)
plt.show()

## 5. Mapa / Visualização Comparativa

Distribuição espacial da população estimada em 2024. Se ``geobr`` estiver disponível, geramos um coropleto; caso contrário, um gráfico de barras horizontal.

In [ ]:
# Dados de 2024 (ou último ano disponível)
df_2024 = df_pop[df_pop["ano"] == df_pop["ano"].max()].copy()

if HAS_GEO:
    # -----------------------------------------------------------------------------
    # Mapa coroplético
    # -----------------------------------------------------------------------------
    gdf = geobr.read_municipality(code_muni="all", year=2022)
    gdf["code_muni"] = gdf["code_muni"].astype(str)
    df_2024["municipio_codigo"] = df_2024["municipio_codigo"].astype(str).str[:6]
    gdf = gdf.merge(df_2024, left_on="code_muni", right_on="municipio_codigo", how="left")
    gdf["populacao"] = gdf["populacao"].fillna(0)

    fig, ax = plt.subplots(figsize=(12, 10))
    gdf.plot(
        column="populacao",
        cmap="YlOrRd",
        linewidth=0.5,
        ax=ax,
        edgecolor="black",
        legend=True,
        legend_kwds={"label": "População estimada (2024)", "shrink": 0.6},
    )
    ax.set_title("População Estimada – Oeste de SC (2024)", fontsize=14, fontweight="bold")
    ax.axis("off")
    fig_path = FIGURES_DIR / "01_mapa_populacao_oeste_sc.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Mapa salvo em: {fig_path}")
    plt.show()
else:
    # -----------------------------------------------------------------------------
    # Gráfico de barras horizontal como fallback
    # -----------------------------------------------------------------------------
    df_2024 = df_2024.sort_values("populacao", ascending=True)
    plt.figure(figsize=(10, 8))
    colors = sns.color_palette("YlOrRd", len(df_2024))
    plt.barh(df_2024["municipio_nome"], df_2024["populacao"], color=colors, edgecolor="black")
    plt.title("População Estimada – Municípios do Oeste de SC (2024)", fontsize=14, fontweight="bold")
    plt.xlabel("População residente estimada")
    plt.ylabel("Município")
    sns.despine()
    fig_path = FIGURES_DIR / "01_barras_populacao_oeste_sc.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Figura salva em: {fig_path}")
    plt.show()

## 6. Indicadores Demográficos Básicos

Cálculo de indicadores para a população venezuelana (stub) e total.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

# Dados reais processados da RI de Chapecó
df = pd.read_parquet('../data/processed/ibge_populacao_estimada_oeste_sc.parquet')
df = df.dropna(subset=['populacao'])
print(f'Municípios: {df["codigo_ibge_7d"].nunique()}, Registros: {len(df)}')
df.head()

## 7. Salvamento de Resultados

Exporta os dados agregados e figuras geradas.

In [ ]:
# -----------------------------------------------------------------------------
# Exportação dos dados agregados
# -----------------------------------------------------------------------------
OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

# Pirâmide etária agregada
piramide_export = pd.concat([
    piramide_ven.assign(grupo="Venezuelana"),
    piramide_total.assign(grupo="Total Oeste SC"),
])
piramide_path = OUTPUT_TABLES / "piramide_etaria_oeste_sc.csv"
piramide_export.to_csv(piramide_path, index=False)
print(f"Pirâmide etária exportada: {piramide_path}")

# Série temporal
ts_path = OUTPUT_TABLES / "serie_temporal_populacao.csv"
df_ts.to_csv(ts_path, index=False)
print(f"Série temporal exportada: {ts_path}")

print("\n--- Execução concluída com sucesso ---")

## 8. Próximos Passos

- [ ] Substituir stub de migrantes por microdados reais do IBGE quando disponíveis;
- [ ] Incluir dados de naturalização (Ministério da Justiça / Polícia Federal);
- [ ] Cruzar com registros administrativos (RAIS, CAGED, SUS);
- [ ] Validar resultados com estatísticas oficiais da OIM e ACNUR.

---

*Notebook elaborado no âmbito do projeto `migracao-venezuelana-oeste-sc`. Os dados de população venezuelana são um stub funcional com estrutura realista, dado o atraso na liberação dos microdados do Censo 2022 pelo IBGE.*